# Can a birthday shape schooling?

**A guided investigation · Chapter 6 · 30–45 minutes · No Python experience needed**

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cmg777/intro2causal/blob/main/notebooks_colab/explore-qob_clean.ipynb)

> **The question**  
> Why might quarter of birth be related to schooling, and what could that teach us about earnings?

People with more schooling often earn more, but that pattern combines education with the reasons people stay in school. Angrist and Krueger investigated a different source of variation: birth timing interacting with school-entry and compulsory-attendance rules. This extract contains men from the 1980 U.S. Census, with schooling, log weekly earnings, and quarter and year of birth. It is the largest file in the collection, so plotting every person at once can conceal rather than reveal structure. We will begin with summaries that use all relevant rows, then compare birth-quarter patterns within cohorts. The differences may be small; reading their size carefully is part of the lesson. A birthday does not automatically become a valid instrument. The investigation ends by asking what assumptions connect birth timing, schooling, and earnings.

### By the end, you will be able to…

- Discover a small schooling pattern within a large census extract and ask what would make it useful for causal reasoning.
- Read and modify three interactive charts using short Python examples.
- Explain one reason a descriptive pattern alone cannot settle this case’s causal question.

### Before you see the evidence

Would you expect people born in different quarters of the same year to have similar schooling?

Write two sentences now. You will return to them at the end.

**Your route:** meet the data → inspect three views → modify and try → ask the causal question.

## Your first few minutes in Colab · 3 minutes

You do not need prior Python experience. A **cell** is one block of text or code. Read the text, then press the **▶** beside the next code cell (or Shift+Enter). Its output appears underneath. Run cells in order: later cells use tables created earlier.

Choose **File → Save a copy in Drive** if you want to keep your changes. If a name is “not defined,” run the earlier setup cells again. **Runtime → Restart session and run all** gives you a fresh start. An error does not mean you have damaged the data: this notebook only reads it.

Keep a short learning log in a text cell. Before each chart, make a prediction. Afterward, write one observation with a number and one question that the chart cannot answer.

**Chart controls:** hover for values, click legend labels to hide/show a group, drag to zoom, and double-click to reset. A chart’s menu also offers a reset/autoscale control.

The setup installs Plotly Express if needed. `import` makes a library available; `pd` and `px` are short names for pandas (tables) and Plotly Express (charts).

In [ ]:
%pip install -q "plotly>=6,<7"

In [ ]:
import pandas as pd
import plotly.express as px
from IPython.display import display

# One consistent chart style; no plotting experience needed.
px.defaults.template = "plotly_white"
px.defaults.color_continuous_scale = "Blues"
px.defaults.color_discrete_sequence = ["#2563eb", "#7c3aed", "#059669", "#64748b"]

## Meet the data · 5 minutes

**One row:** Man in U.S. Census sample.  
**Stored size:** 329,509 rows × 9 columns.

| Column | What it records |
|---|---|
| `s` | Completed years of schooling. |
| `lnw` | Natural logarithm of weekly earnings; not dollar earnings. |
| `qob` | Quarter of birth: 1 January–March through 4 October–December. |
| `yob` | Year of birth; two-digit 1930s codes in qob_clean, full years in childlabor_clean. |

lnw is log weekly earnings, not dollars. yob uses two-digit birth-year codes. Group means use all available rows; any display sample must be identified.

`read_csv` downloads a table. We call it `df`. `head(5)` shows just five rows, not a separate sample used for the analysis. Square brackets select columns by their names.

In [ ]:
DATA_URL = "https://raw.githubusercontent.com/cmg777/intro2causal/main/data/ch6/qob_clean.csv"
df = pd.read_csv(DATA_URL)
print(f"Loaded {len(df):,} rows and {len(df.columns)} columns")
display(df[['s', 'lnw', 'qob', 'yob']].head(5))

### Pause: what do the rows mean?

Choose one displayed row. Translate it into a sentence using the variable guide. Which values are categories or identifiers? Which are quantities? A numeric storage type does not necessarily make a code a meaningful measurement.

The next table checks types, missing values, and the number of distinct values for the key columns. `isna()` marks missing cells; `sum()` counts them. Missing is not zero. Ask which comparison would lose a row if one of its required values were missing.

In [ ]:
key_columns = ['s', 'lnw', 'qob', 'yob']
checks = pd.DataFrame({"storage_type": df[key_columns].dtypes.astype(str),
                       "missing_rows": df[key_columns].isna().sum(),
                       "distinct_values": df[key_columns].nunique()})
display(checks)
print("The charts below omit missing values required for each plotted quantity.")

### A small summary before plotting

`describe()` summarizes numeric columns: `count` counts available values, `mean` is the average, and `50%` is the median. The median splits ordered values into two halves. Averages of identifiers or arbitrary category codes are not useful descriptions, even if Python calculates them.

For category-like key variables, the second table counts values, including missing entries. Check these labels before interpreting a chart.

In [ ]:
numeric_keys = df[key_columns].select_dtypes(include="number").columns
if len(numeric_keys):
    display(df[numeric_keys].describe().round(2))
for column in key_columns:
    if df[column].nunique() <= 8:
        display(df[column].value_counts(dropna=False).rename("Rows").to_frame())

## Read the chart code · 2 minutes

`px.bar`, `px.scatter`, `px.line`, and `px.box` choose a chart type. `x` and `y` name columns for the axes; `color` distinguishes groups or values. `labels` translates column names for readers. `fig.show()` displays the result.

A table transformation and a plot answer different questions: first decide what each plotted row represents, then choose how to display it. Before running each chart, predict what you expect to see.

In [ ]:
# Display labels: run once; you do not need to memorize this mapping.
labels = {'ability_group': 'Ability group',
 'age': 'Age (years)',
 'census': 'Census year',
 'group': 'State group',
 'lnw': 'Mean log weekly earnings',
 'offer': 'Scholarship assignment',
 'plan': 'Assigned insurance plan',
 'qob': 'Quarter of birth',
 's': 'Years of schooling',
 'state_name': 'Simulated state',
 'yob': 'Birth year'}

## Visual investigation 1 · The familiar schooling gradient

**Predict:** How do average log wages vary with schooling?

We summarize the full file by completed years of schooling. lnw is a natural logarithm: differences are on a proportional scale, not dollar differences. We do not exponentiate mean logs and call that mean earnings.

In [ ]:
school = df.groupby("s", as_index=False).agg(lnw=("lnw", "mean"), people=("lnw", "count"))
fig = px.scatter(school, x="s", y="lnw", size="people", hover_data=["people"], labels=labels, title="Schooling and wages in the census extract")
fig.show()

**Explore the chart:** Hover over common schooling levels. A larger circle means more available records, not a larger causal effect.

**Write before reading on:** Describe one pattern with its unit and comparison group. Then give one possible explanation.

<details>
<summary>Interpretation checkpoint</summary>

The upward association includes the effects of selection into schooling. Grouping makes the pattern readable but does not remove confounding.

</details>

## Visual investigation 2 · Compare birthdays within a cohort

**Predict:** Are birth-quarter schooling differences visible within birth years?

Grouping on both birth year and quarter produces one average for each combination. Keeping cohorts separate avoids attributing broad cohort differences to birth quarter.

In [ ]:
cohorts = df.groupby(["yob", "qob"], as_index=False)[["s", "lnw"]].mean()
cohorts["birth_year"] = 1900 + cohorts["yob"]
cohorts["quarter"] = "Q" + cohorts["qob"].astype(str)
fig = px.line(cohorts, x="birth_year", y="s", color="quarter", markers=True, labels={"birth_year": "Birth year", "s": "Mean years of schooling", "quarter": "Birth quarter"}, title="Small schooling differences within cohorts")
fig.show()

**Explore the chart:** Toggle Q2 and Q3 off to compare Q1 and Q4. Use hover to read differences that the axis may magnify.

**Write before reading on:** Describe one pattern with its unit and comparison group. Then give one possible explanation.

<details>
<summary>Interpretation checkpoint</summary>

A small schooling difference may motivate an instrument, but small differences also demand careful inference. Birth timing is not automatically random with respect to every earnings determinant.

</details>

## Visual investigation 3 · Follow the link to earnings

**Predict:** Do the same quarter patterns appear in average log wages?

We reuse the same cohort-quarter table and change the outcome. This keeps the groups comparable while asking a new question.

In [ ]:
fig = px.line(cohorts, x="birth_year", y="lnw", color="quarter", markers=True, labels={"birth_year": "Birth year", "lnw": "Mean log weekly earnings", "quarter": "Birth quarter"}, title="The outcome link is a separate question")
fig.show()

**Explore the chart:** Look at both the common cohort pattern and the within-cohort quarter differences. Which variation is larger?

**Write before reading on:** Describe one pattern with its unit and comparison group. Then give one possible explanation.

<details>
<summary>Interpretation checkpoint</summary>

The instrument argument requires a connection to schooling and a reason to rule out other routes from birth timing to earnings. These charts establish neither assumption by themselves.

</details>

## Make it yours · 8 minutes

### 1. Modify an example

Change the outcome in the first summary to median log wages using median instead of mean.

Copy that chart cell before editing, so you can compare the original and modified outputs. Say what changed in your question as well as in your code.

### 2. Try a small investigation

For men born in 1930 (yob == 30), plot mean schooling by quarter of birth.

Write a prediction first. The optional starter cell below is safe to leave unchanged; later cells do not depend on your answer.

In [ ]:
# Your investigation goes here.
# Describe your expected pattern before writing code.

<details>
<summary>Hint 1 · choose the right evidence</summary>

Filter first so a birth-quarter comparison is made inside one cohort.

</details>

<details>
<summary>Hint 2 · a Python starting point</summary>

Then group by qob and average s.

</details>

<details>
<summary>Optional worked solution · open after trying</summary>

<pre><code>cohort = df[df[&quot;yob&quot;] == 30].groupby(&quot;qob&quot;, as_index=False)[&quot;s&quot;].mean()
fig = px.bar(cohort, x=&quot;qob&quot;, y=&quot;s&quot;, labels=labels, title=&quot;Birth quarter within the 1930 cohort&quot;)
fig.show()</code></pre>

Read the numerical size of the difference and the bar baseline. A birthday pattern is motivation for investigating assumptions, not proof of an education effect.

Copy the code into the starter cell to run it.

</details>

## What can you now say? · 4 minutes

Return to your initial prediction. Write a four-sentence evidence note:

1. **I expected…** State your prediction.
2. **I observed…** Use a chart, a comparison, and the correct units.
3. **This does not establish…** Name a causal claim the evidence cannot settle.
4. **To learn more, I would need…** Propose an additional measurement or a more credible comparison.

### Carry this question into the chapter

**Besides changing schooling, could quarter of birth be related to earnings through another route?**

[Continue with Chapter 6: The Wages of Schooling](https://cmg777.github.io/intro2causal/book/_book/notebooks_quarto/06-wages-of-schooling.html#strategy-2-quarter-of-birth-iv) · [Return to the case collection](https://cmg777.github.io/intro2causal/data-dictionary.html#ch6-qob-clean-csv)

### Source and preparation notes

[Angrist and Krueger (1991), Does Compulsory School Attendance Affect Schooling and Earnings? Author data archive](https://economics.mit.edu/people/faculty/josh-angrist/angrist-data-archive)

Prepared extract from Angrist and Krueger's 1980 Census data. Notebook analysis restricts birth cohorts as described there; the full source construction is not reproduced in this repository.

These exploratory charts are based on the committed CSV. They are not a complete replication of the original study. Keep the source study, this prepared extract, and your descriptive results distinct.